# Medical Synthetic Image Training Notebook

This notebook shows how to:
- Load medical images
- Generate synthetic images
- Train CNN
- Compare accuracy

In [1]:

!uv pip install medigan torch torchvision opencv-python


Resolved 54 packages in 1.91s                                        
   Building medigan==1.0.0                                             
   Building medigan==1.0.0                                     
⠙ Preparing packages... (0/8)
   Building medigan==1.0.00m     0 B/22.46 KiB                     
⠙ Preparing packages... (0/8)
   Building medigan==1.0.00m 14.88 KiB/22.46 KiB                   
⠙ Preparing packages... (0/8)
pyjwt      ------------------------------ 14.88 KiB/22.46 KiB
   Building medigan==1.0.00m     0 B/47.04 KiB                     
⠙ Preparing packages... (0/8)
pyjwt      ------------------------------ 14.88 KiB/22.46 KiB
   Building medigan==1.0.00m 14.88 KiB/47.04 KiB                   
⠙ Preparing packages... (0/8)
pyjwt      ------------------------------ 14.88 KiB/22.46 KiB
pycparser  ------------------------------ 14.88 KiB/47.04 KiB
   Building medigan==1.0.0     0 B/4.27 MiB                    
⠙ Preparing packages... (0/8)
pyjwt      -------------------

# Task
Download the 'Chest X-Ray Images (Pneumonia)' dataset from Kaggle.

## Download Chest X-Ray Dataset

### Subtask:
Download the 'Chest X-Ray Images (Pneumonia)' dataset from Kaggle.


**Reasoning**:
The first step to download a Kaggle dataset is to install the Kaggle API client. This is done using `!pip install kaggle`.



In [6]:
!uv pip install kaggle kagglehub

Resolved 29 packages in 151ms                                        
Installed 2 packages in 3ms                                 
 + kagglehub==0.4.1
 + pyyaml==6.0.3


**Reasoning**:
After installing the Kaggle API client, the next step is to upload the Kaggle API token (`kaggle.json`) to the Colab environment and set its permissions, as per the instructions. The `kaggle.json` file needs to be moved to the `~/.kaggle/` directory and its permissions set to `600` for security.



The previous step failed with the error "mv: cannot stat 'kaggle.json': No such file or directory". This indicates that the `kaggle.json` file, which contains your Kaggle API token, was not found in the expected location.

Please ensure you have uploaded your `kaggle.json` file to the Colab environment. You can do this by clicking the 'Files' icon on the left sidebar (folder icon), then 'Upload to session storage' (file with up arrow icon), and navigating to your `kaggle.json` file. It should be uploaded directly into the root directory of your Colab environment or manually moved to `~/.kaggle/`.

Once the `kaggle.json` file is uploaded, please **re-run the previous code cell** to move it to the correct directory (`~/.kaggle/`) and set its permissions.

**Reasoning**:
After the `kaggle.json` file is correctly placed and its permissions are set (by the user re-running the previous cell), the next logical step in the subtask is to download the specified Kaggle dataset.



## Download Chest X-Ray Dataset via KaggleHub

### Subtask:
Use the `kagglehub` library to download the 'Chest X-Ray Images (Pneumonia)' dataset. This method typically handles authentication automatically in Colab.


**Reasoning**:
The subtask is to download the dataset using `kagglehub`. The first step is to import the `kagglehub` library and then use its `dataset_download` function to get the dataset.



In [7]:
import kagglehub
import os

print("Downloading 'Chest X-Ray Images (Pneumonia)' dataset from KaggleHub...")

# Define the Kaggle dataset handle
dataset_handle = "paultimothymooney/chest-xray-pneumonia"

# Download the dataset using kagglehub
download_path = kagglehub.dataset_download(dataset_handle)

print(f"Dataset downloaded successfully to: {download_path}")

# Assuming the dataset is downloaded to a directory named after the dataset handle
# The actual files are often nested within another directory. We need to find the actual directory.
# Let's verify the path and contents if possible.

# Since kagglehub.dataset_download returns the path to the extracted files/directory,
# we can expect the 'chest_xray' folder (or similar structure) to be directly under it.

# For simplicity, let's assume the main folder is directly under download_path
# This often needs adjustment based on how the specific Kaggle dataset is structured inside its zip.
# A common structure is `download_path/dataset_name/data/` or `download_path/data/`

# Let's inspect the contents of the download_path to determine the actual dataset root
# This part is for debugging/verification and might need refinement depending on the actual download structure.
print("Contents of the downloaded directory:")
for root, dirs, files in os.walk(download_path):
    level = root.replace(download_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")
    if level >= 2: # Limit depth for printout to avoid too much output
        break


/home/vrathik/grad_cam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.29G/2.29G [02:35<00:00, 15.8MB/s]

Extracting files...


Dataset downloaded successfully to: /home/vrathik/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2
Contents of the downloaded directory:
2/
    chest_xray/
        chest_xray/
            .DS_Store


**Reasoning**:
The dataset has been downloaded and implicitly unzipped by `kagglehub`. Now, to complete the subtask of verifying the resulting directory structure and prepare for subsequent steps, I need to identify the exact root directory of the extracted dataset (i.e., where 'train', 'test', 'val' folders reside) and print its contents.



In [8]:
import os

# The download_path from the previous cell is '/kaggle/input/chest-xray-pneumonia'
# Based on the previous os.walk output, the actual dataset is nested further.
# The 'chest_xray' directory containing 'train', 'test', 'val' is at:
# /kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray/

dataset_root = os.path.join(download_path, "chest_xray", "chest_xray")

print(f"Verifying the dataset structure at: {dataset_root}")

# List the contents of the determined dataset_root
if os.path.exists(dataset_root):
    print("Contents:")
    for item in os.listdir(dataset_root):
        full_path = os.path.join(dataset_root, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}/")
            # For directories like train/test/val, list their immediate subdirectories (NORMAL/PNEUMONIA)
            if item in ['train', 'test', 'val']:
                for sub_item in os.listdir(full_path):
                    print(f"    [DIR] {sub_item}/")
        else:
            print(f"  [FILE] {item}")
else:
    print(f"Error: Dataset root directory '{dataset_root}' not found.")

# Set the real_path for the original notebook based on the kagglehub download
# This path will be used in subsequent steps of the original notebook.
real_path_base = dataset_root

# Print the base path that will be used for the 'real' dataset
print(f"\nSet 'real_path_base' for the original dataset to: {real_path_base}")

Verifying the dataset structure at: /home/vrathik/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray
Contents:
  [DIR] test/
    [DIR] .DS_Store/
    [DIR] NORMAL/
    [DIR] PNEUMONIA/
  [DIR] train/
    [DIR] .DS_Store/
    [DIR] NORMAL/
    [DIR] PNEUMONIA/
  [FILE] .DS_Store
  [DIR] val/
    [DIR] .DS_Store/
    [DIR] NORMAL/
    [DIR] PNEUMONIA/

Set 'real_path_base' for the original dataset to: /home/vrathik/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/chest_xray


In [19]:
from medigan import Generators
import cv2, os

mg = Generators()
images = mg.generate(model_id="00019_PGGAN_CHEST_XRAY", num_samples=200, output_path="synthetic/PNEUMONIA")

print("Synthetic images generated:", len(images))


File is not a zip file


ERROR:root:00019_PGGAN_CHEST_XRAY: This model could not be added to model_executor list: ('Connection broken: IncompleteRead(170248408 bytes read, 127779503 more expected)', IncompleteRead(170248408 bytes read, 127779503 more expected))


ChunkedEncodingError: ('Connection broken: IncompleteRead(170248408 bytes read, 127779503 more expected)', IncompleteRead(170248408 bytes read, 127779503 more expected))

In [18]:
from medigan import Generators
import cv2, os

# List available models
mg = Generators()
available_models = list(mg.config_manager.config_dict.keys())
print("Available models:", available_models)


Available models: ['00001_DCGAN_MMG_CALC_ROI', '00002_DCGAN_MMG_MASS_ROI', '00003_CYCLEGAN_MMG_DENSITY_FULL', '00004_PIX2PIX_MMG_MASSES_W_MASKS', '00005_DCGAN_MMG_MASS_ROI', '00006_WGANGP_MMG_MASS_ROI', '00007_INPAINT_BRAIN_MRI', '00008_C-DCGAN_MMG_MASSES', '00009_PGGAN_POLYP_PATCHES_W_MASKS', '00010_FASTGAN_POLYP_PATCHES_W_MASKS', '00011_SINGAN_POLYP_PATCHES_W_MASKS', '00012_C-DCGAN_MMG_MASSES', '00013_CYCLEGAN_MMG_DENSITY_OPTIMAM_MLO', '00014_CYCLEGAN_MMG_DENSITY_OPTIMAM_CC', '00015_CYCLEGAN_MMG_DENSITY_CSAW_MLO', '00016_CYCLEGAN_MMG_DENSITY_CSAW_CC', '00017_DCGAN_XRAY_LUNG_NODULES', '00018_WGANGP_XRAY_LUNG_NODULES', '00019_PGGAN_CHEST_XRAY', '00020_PGGAN_CHEST_XRAY', '00021_CYCLEGAN_BRAIN_MRI_T1_T2', '00022_WGAN_CARDIAC_AGING', '00023_PIX2PIXHD_BREAST_DCEMRI']


In [ ]:

import shutil, os

real_path = os.path.join(dataset_root, "train", "PNEUMONIA")
synth_path = "synthetic/PNEUMONIA"

if os.path.exists(real_path):
    for f in os.listdir(synth_path):
        shutil.copy(os.path.join(synth_path,f), real_path)
else:
    print(f"Directory not found: {real_path}")

print("Synthetic images merged with real dataset")


In [ ]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(os.path.join(dataset_root, "train"), transform=transform)
test_data = datasets.ImageFolder(os.path.join(dataset_root, "test"), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)


In [ ]:

import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,16,3,1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32*54*54,128),
            nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self,x):
        return self.net(x)


In [ ]:

import torch, torch.optim as optim

model = CNN()
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    for x,y in train_loader:
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out,y)
        loss.backward()
        opt.step()
    print("Epoch:",epoch,"Loss:",loss.item())


In [ ]:

correct,total=0,0
model.eval()

with torch.no_grad():
    for x,y in test_loader:
        out = model(x)
        pred = out.argmax(1)
        correct += (pred==y).sum().item()
        total += y.size(0)

print("Test Accuracy:",correct/total)
